# Funhouse Agent — Interactive Test Notebook

Test the GeotechAgent in two modes:
1. **Scripted scenarios** — Run predefined engineering questions and check results
2. **Interactive chat** — Use the NotebookChat widget for freeform Q&A

Works with **PrompterAPI** (Databricks/Funhouse) or **ClaudeEngine** (local).

## 1. Setup

Choose your engine below. In **Databricks/Funhouse**, use PrompterAPI.
Locally, use ClaudeEngine (requires `ANTHROPIC_API_KEY` env var).

In [ ]:
# ── Pick ONE engine ──────────────────────────────────────────────

# Option A: PrompterAPI (Databricks / Funhouse)
# from funhouse.prompter import PrompterAPI
# engine = PrompterAPI()

# Option B: Claude (local, needs ANTHROPIC_API_KEY)
from funhouse_agent import ClaudeEngine
engine = ClaudeEngine()

# Verify engine
print(f"Engine: {type(engine).__name__}")
print(f"Has chat: {hasattr(engine, 'chat')}")
print(f"Has vision: {hasattr(engine, 'analyze_image')}")

In [ ]:
# ── Create agent ─────────────────────────────────────────────────
from funhouse_agent import GeotechAgent

agent = GeotechAgent(
    genai_engine=engine,
    max_rounds=15,
    verbose=True,       # shows round-by-round progress
)
print("Agent ready.")

## 2. Scripted Scenarios

Run predefined engineering questions and inspect results. Each cell is a standalone scenario — run them in order or pick and choose.

### 2a. Bearing Capacity — Basic Analysis + Follow-Up

In [ ]:
agent.reset()

r1 = agent.ask(
    "Calculate the ultimate and allowable bearing capacity of a 2m wide strip footing "
    "embedded 1.5m deep in sand with friction angle 30 degrees and unit weight 18 kN/m3. "
    "Use Vesic method with FOS=3."
)
print("=== Answer ===")
print(r1.answer)
print(f"\nRounds: {r1.rounds}, Time: {r1.total_time_s:.1f}s, Tools: {[t['tool_name'] for t in r1.tool_calls]}")

In [ ]:
# Follow-up: same conversation context
r2 = agent.ask("Now what if the groundwater table is at the surface? How does that change the result?")
print("=== Follow-Up Answer ===")
print(r2.answer)
print(f"\nRounds: {r2.rounds}, Time: {r2.total_time_s:.1f}s")

### 2b. Slope Stability + Grid Search

In [ ]:
agent.reset()

r = agent.ask(
    "Analyze the stability of a 10m high 2H:1V slope in homogeneous clay with "
    "c=25 kPa, phi=10 degrees, gamma=18 kN/m3. Use Bishop method with a circular "
    "slip surface centered at x=5, y=15 with radius 18m. "
    "Ground surface: (0,10) to (20,10) to (40,0) to (60,0)."
)
print("=== Answer ===")
print(r.answer)

In [ ]:
# Follow-up: search for critical surface
r2 = agent.ask("Now search for the critical circular slip surface using a grid search. What is the minimum FOS?")
print("=== Answer ===")
print(r2.answer)

### 2c. Multi-Module Workflow: Foundation Design → Settlement → Calc Package

In [ ]:
agent.reset()

# Q1: Size a footing for a given column load
r1 = agent.ask(
    "I need to design a foundation for a column load of 800 kN. The site has medium dense "
    "sand with phi=32, gamma=18 kN/m3, and Es=25000 kPa at a depth of 1.5m. "
    "First, what size square footing do I need for a factor of safety of 3?"
)
print("=== Q1: Footing Size ===")
print(r1.answer)
print(f"\nRounds: {r1.rounds}, Tools: {[t['tool_name'] for t in r1.tool_calls]}")

In [ ]:
# Q2: Settlement check (follow-up, same conversation)
r2 = agent.ask("Using that footing size, estimate the immediate settlement. Is it within the 25mm limit?")
print("=== Q2: Settlement Check ===")
print(r2.answer)
print(f"\nRounds: {r2.rounds}, Tools: {[t['tool_name'] for t in r2.tool_calls]}")

In [ ]:
# Q3: Generate HTML calc package (follow-up, same conversation)
r3 = agent.ask(
    "Now generate an HTML calc package for the bearing capacity analysis with "
    "project name 'Column C-14 Foundation' and save to output/foundation_bc.html."
)
print("=== Q3: Calc Package ===")
print(r3.answer)
print(f"\nRounds: {r3.rounds}, Tools: {[t['tool_name'] for t in r3.tool_calls]}")

### 2d. Calc Package Generation — HTML + PDF with Figures

These scenarios generate standalone calc packages. Check that:
- HTML files contain embedded base64 PNG figures (not text-based diagrams)
- PDF generation works (requires LaTeX) or gracefully reports unavailability

In [ ]:
agent.reset()

r = agent.ask(
    "Generate an HTML calculation package for a bearing capacity analysis of a 2.5m square "
    "footing at 1.5m depth in sand with phi=35, gamma=18.5 kN/m3. "
    "Project name: 'Highway Bridge B-42', project number 'P2024-001', "
    "engineer 'S. O\\'Connell'. Save to output/bc_test.html."
)
print("=== Answer ===")
print(r.answer)
print(f"\nRounds: {r.rounds}, Tools: {[t['tool_name'] for t in r.tool_calls]}")

In [ ]:
# Verify HTML calc package has real figures (base64 PNG, not text)
import os
html_path = "output/bc_test.html"
if os.path.exists(html_path):
    with open(html_path, encoding="utf-8") as f:
        html = f.read()
    size_kb = len(html) / 1024
    n_figures = html.count("data:image/png;base64,")
    print(f"File size: {size_kb:.1f} KB")
    print(f"Embedded PNG figures: {n_figures}")
    if n_figures > 0:
        print("✓ Real matplotlib figures detected")
    else:
        print("✗ No embedded figures — check matplotlib availability")
else:
    print(f"File not found: {html_path}")

In [ ]:
# Slope stability HTML calc package
agent.reset()

r = agent.ask(
    "Create an HTML calc package for slope stability analysis of an 8m high, 2.5H:1V slope "
    "in clay with c=30 kPa, phi=12 degrees, gamma=17.5 kN/m3. Use Bishop method. "
    "Surface from (0,8) to (20,8) to (40,0) to (60,0). "
    "Project: 'Levee Segment LS-3', number 'P2024-003'. Save to output/slope_test.html."
)
print("=== Answer ===")
print(r.answer)

In [ ]:
# Verify slope HTML has figures
html_path = "output/slope_test.html"
if os.path.exists(html_path):
    with open(html_path, encoding="utf-8") as f:
        html = f.read()
    size_kb = len(html) / 1024
    n_figures = html.count("data:image/png;base64,")
    print(f"File size: {size_kb:.1f} KB")
    print(f"Embedded PNG figures: {n_figures}")
    if n_figures > 0:
        print("✓ Real matplotlib figures detected")
    else:
        print("✗ No embedded figures")
else:
    print(f"File not found: {html_path}")

### 2e. Seismic + Liquefaction

In [ ]:
agent.reset()

r1 = agent.ask(
    "Classify a site with average shear wave velocity Vs30=250 m/s. "
    "What are the site coefficients for Ss=1.0g and S1=0.4g?"
)
print("=== Site Classification ===")
print(r1.answer)

r2 = agent.ask(
    "Evaluate liquefaction triggering for a sand layer at 5m depth with N160=12, "
    "fines content 15%, unit weight 19 kN/m3, PGA=0.3g, M=7.5, GWT at 2m."
)
print("\n=== Liquefaction ===")
print(r2.answer)

## 3. Interactive Chat (NotebookChat Widget)

Use the chat widget below for freeform Q&A. Type engineering questions and the agent will use its tools to answer.

**Try these prompts:**
- "What is the bearing capacity of a 3m footing on clay with cu=60 kPa?"
- "Analyze a 6m cantilever retaining wall with phi=30 backfill"
- "Generate an HTML calc package for the last analysis"
- Attach an image: `chat.attach("path/to/cross_section.png")` then ask about it

In [ ]:
from funhouse_agent.notebook import NotebookChat

# Reset agent for a fresh interactive session
agent.reset()

chat = NotebookChat(agent, height="500px")
chat.display()

In [ ]:
# Attach an image for vision analysis (optional)
# chat.attach("path/to/cross_section.png")
# Then type a question about the image in the chat widget above

## 4. Output File Review

After running scenarios, inspect generated output files.

In [ ]:
# List all output files generated during this session
from pathlib import Path

output_dir = Path("output")
if output_dir.exists():
    files = sorted(output_dir.glob("*"))
    print(f"Output files ({len(files)}):")
    for f in files:
        size = f.stat().st_size
        print(f"  {f.name:40s} {size:>10,} bytes")
else:
    print("No output directory yet — run a calc package scenario first.")

In [ ]:
# Preview an HTML calc package inline (renders in notebook)
from IPython.display import HTML, display as ipy_display

html_file = "output/bc_test.html"  # change to any output file
if os.path.exists(html_file):
    with open(html_file, encoding="utf-8") as f:
        html_content = f.read()
    # Show in a scrollable iframe
    iframe = f'<iframe srcdoc="{html_content.replace(chr(34), "&quot;")}" width="100%" height="600px" style="border:1px solid #ccc;"></iframe>'
    ipy_display(HTML(iframe))
else:
    print(f"File not found: {html_file} — run the bearing capacity calc package scenario first.")

In [ ]:
# Use NotebookChat's preview_file for output inspection
# (shows HTML inline or PDF download link)
if hasattr(chat, 'output_files') and chat.output_files:
    print(f"Files from chat session: {chat.output_files}")
    for fpath in chat.output_files:
        chat.preview_file(fpath)
else:
    print("No output files from interactive chat session yet.")